# Carrier & Vendor Performance Scorecard (DataCo Smart Supply Chain)
**Tools:** Python, Excel (formulas + conditional formatting), Tableau
**Goal:** score shipping modes, markets, regions and departments on on-time delivery, delay and profit, then flag who is below target.

DataCo has no carrier-name column, so **Shipping Mode** (Standard / Second / First Class / Same Day) is the service-level "carrier" view. Markets, regions, departments and customer segments are the "vendor / segment" views.

Every resume number must come from the **"Resume metrics"** cell. Run the cells top to bottom.

## 1. Load the data
Use **Option A** *or* **Option B**, not both.

In [ ]:
# OPTION A: download directly in Colab
import kagglehub
DATA_DIR = kagglehub.dataset_download("shashwatwork/dataco-smart-supply-chain-for-big-data-analysis")
print("Path to dataset files:", DATA_DIR)

In [ ]:
# OPTION B: upload the zip you downloaded to your PC (skip if Option A worked)
# from google.colab import files
# import zipfile, os
# up = files.upload()
# zname = list(up.keys())[0]
# DATA_DIR = "/content/dataco"
# os.makedirs(DATA_DIR, exist_ok=True)
# zipfile.ZipFile(zname).extractall(DATA_DIR)
# print(os.listdir(DATA_DIR))

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

hits = glob.glob(os.path.join(DATA_DIR, "**", "DataCoSupplyChainDataset.csv"), recursive=True)
assert hits, "DataCoSupplyChainDataset.csv not found - check DATA_DIR and the zip contents"
raw = pd.read_csv(hits[0], encoding="latin-1")   # the file is not UTF-8
print(raw.shape)
raw.head(3)

## 2. Clean column names and check the columns we need

In [ ]:
def norm(c):
    return re.sub(r"[^a-z0-9]+", "_", c.lower()).strip("_")

df = raw.copy()
df.columns = [norm(c) for c in df.columns]

NEEDED = ["order_id", "shipping_mode", "market", "order_region", "department_name",
          "category_name", "customer_segment", "delivery_status", "order_status",
          "days_for_shipping_real", "days_for_shipment_scheduled",
          "order_date_dateorders", "sales", "order_profit_per_order"]
missing = [c for c in NEEDED if c not in df.columns]
assert not missing, f"Missing columns: {missing}. Available: {list(df.columns)}"
print("All needed columns found.")

## 3. Data quality checks
The file has **one row per order item**, so a multi-item order appears several times. On-time % must be measured per **order**, otherwise big orders count more than once.

In [ ]:
print("rows (order items):", len(df), "| unique orders:", df.order_id.nunique())
print()
print(df.delivery_status.value_counts())
print()
print(df.order_status.value_counts())
chk = df.groupby("order_id")[["shipping_mode", "days_for_shipping_real", "days_for_shipment_scheduled"]].nunique()
print()
print("orders with inconsistent shipping fields across items:", int((chk > 1).any(axis=1).sum()))
na = df[NEEDED].isna().sum()
print("columns with missing values:", na[na > 0].to_dict())

## 4. Build one table with one row per order
Rules (state them in your README):
- One row per order. Shipping fields come from the first item. **Sales and profit are summed across items.**
- Excluded: canceled shipments, `CANCELED` and `SUSPECTED_FRAUD` orders.
- **Late** = actual shipping days > scheduled shipping days. This is cross-checked against the dataset's own `Delivery Status`.
- Department and category come from the order's first item.

In [ ]:
orders = df.groupby("order_id", as_index=False).agg(
    shipping_mode=("shipping_mode", "first"),
    market=("market", "first"),
    order_region=("order_region", "first"),
    department_name=("department_name", "first"),
    category_name=("category_name", "first"),
    customer_segment=("customer_segment", "first"),
    delivery_status=("delivery_status", "first"),
    order_status=("order_status", "first"),
    days_real=("days_for_shipping_real", "first"),
    days_sched=("days_for_shipment_scheduled", "first"),
    order_date_raw=("order_date_dateorders", "first"),
    sales=("sales", "sum"),
    profit=("order_profit_per_order", "sum"),
)
n_before = len(orders)
excluded = (orders.delivery_status.astype(str).str.lower() == "shipping canceled") | \
           (orders.order_status.astype(str).str.upper().isin(["CANCELED", "SUSPECTED_FRAUD"]))
orders = orders[~excluded].copy()
print(f"orders before: {n_before:,} | excluded: {int(excluded.sum()):,} | analyzed: {len(orders):,}")

orders["is_late"] = (orders.days_real > orders.days_sched).astype(int)
orders["delay_days"] = orders.days_real - orders.days_sched
orders["order_date"] = pd.to_datetime(orders.order_date_raw, errors="coerce")
orders["month"] = orders.order_date.dt.to_period("M").astype(str)

agree = ((orders.delivery_status.astype(str).str.lower() == "late delivery").astype(int) == orders.is_late).mean()
print(f"agreement between my is_late and the dataset's Delivery Status: {100*agree:.1f}%")
print("unparseable order dates:", int(orders.order_date.isna().sum()))

## 5. Overall performance

In [ ]:
overall = pd.DataFrame({
    "orders": [len(orders)],
    "late_orders": [int(orders.is_late.sum())],
    "on_time_pct": [round(100 * (1 - orders.is_late.mean()), 2)],
    "avg_ship_days": [round(orders.days_real.mean(), 2)],
    "avg_scheduled_days": [round(orders.days_sched.mean(), 2)],
    "avg_delay_when_late": [round(orders.loc[orders.is_late == 1, "delay_days"].mean(), 2)],
})
overall

## 6. Scorecards
Settings you can change: the on-time target, the amber band, and the minimum orders a segment needs to be judged.

**RAG status:** GREEN = at or above target. AMBER = within the amber band below target. RED = below that.
If everything comes out RED, the target is above what this data can reach. Say so in your write-up and also rank segments relative to each other.

In [ ]:
TARGET_ON_TIME = 90   # percent
AMBER_BAND = 10       # points below target that still count as AMBER
MIN_ORDERS = 200      # segments smaller than this are not ranked

def scorecard(d, by):
    g = d.groupby(by)
    out = g.agg(orders=("order_id", "count"),
                late_orders=("is_late", "sum"),
                avg_ship_days=("days_real", "mean"),
                avg_sched_days=("days_sched", "mean"),
                sales=("sales", "sum"),
                profit=("profit", "sum")).reset_index()
    delay = d[d.is_late == 1].groupby(by)["delay_days"].mean().rename("avg_delay_when_late")
    out = out.merge(delay, on=by, how="left")
    out["on_time_pct"] = 100 * (1 - out.late_orders / out.orders)
    out["rag"] = np.where(out.on_time_pct >= TARGET_ON_TIME, "GREEN",
                 np.where(out.on_time_pct >= TARGET_ON_TIME - AMBER_BAND, "AMBER", "RED"))
    return out.sort_values("on_time_pct").round(2).reset_index(drop=True)

DIMS = {
    "shipping_mode": "Shipping Mode",
    "market": "Market",
    "order_region": "Region",
    "department_name": "Department",
    "customer_segment": "Customer Segment",
}
SC = {by: scorecard(orders, by) for by in DIMS}
display(SC["shipping_mode"])

In [ ]:
display(SC["market"])
display(SC["order_region"])

In [ ]:
display(SC["department_name"])
display(SC["customer_segment"])

### Best vs worst performers (segments with enough orders)

In [ ]:
rows = []
for by, label in DIMS.items():
    s = SC[by].query("orders >= @MIN_ORDERS")
    if s.empty:
        continue
    worst, best = s.iloc[0], s.iloc[-1]
    rows.append({
        "dimension": label,
        "segments_ranked": len(s),
        "worst": worst[by], "worst_on_time_pct": worst.on_time_pct,
        "best": best[by], "best_on_time_pct": best.on_time_pct,
        "gap_points": round(best.on_time_pct - worst.on_time_pct, 2),
        "below_target": int((s.on_time_pct < TARGET_ON_TIME).sum()),
        "red": int((s.rag == "RED").sum()),
    })
gaps = pd.DataFrame(rows)
gaps

In [ ]:
sm = SC["shipping_mode"]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(sm["shipping_mode"], sm["on_time_pct"])
ax.axvline(TARGET_ON_TIME, linestyle="--")
ax.set_xlabel("On-time delivery (%)"); ax.set_xlim(0, 100)
ax.set_title(f"On-time % by shipping mode (dashed = {TARGET_ON_TIME}% target)")
plt.tight_layout(); plt.show()

### How late are late orders, and what does lateness cost?

In [ ]:
print(orders.loc[orders.is_late == 1, "delay_days"].describe().round(2))

late_vs_ontime = orders.groupby("is_late").agg(
    orders=("order_id", "count"),
    avg_sales=("sales", "mean"),
    avg_profit=("profit", "mean"),
).round(2)
late_vs_ontime.index = ["on time", "late"]
display(late_vs_ontime)

In [ ]:
trend = orders[orders.month != "NaT"].groupby("month").agg(
    orders=("order_id", "count"), late=("is_late", "mean")).reset_index()
trend["on_time_pct"] = (100 * (1 - trend.late)).round(2)
trend = trend[trend.orders >= 100]
display(trend[["month", "orders", "on_time_pct"]])
trend.plot(x="month", y="on_time_pct", figsize=(9, 3.5), marker="o", legend=False)
plt.ylabel("On-time (%)"); plt.xticks(rotation=60); plt.title("Monthly on-time %"); plt.tight_layout(); plt.show()

## 7. Resume metrics
Copy numbers **from this output only**.

In [ ]:
o = overall.iloc[0]
print(f"Orders analyzed: {int(o.orders):,}")
print(f"Overall on-time: {o.on_time_pct}% | late: {round(100 - o.on_time_pct, 2)}% | avg delay when late: {o.avg_delay_when_late} days")
print(f"Avg actual shipping days: {o.avg_ship_days} vs scheduled {o.avg_scheduled_days}")
print()
print("Shipping modes:")
print(SC["shipping_mode"][["shipping_mode", "orders", "on_time_pct", "avg_delay_when_late", "rag"]].to_string(index=False))
print()
for _, r in gaps.iterrows():
    print(f"{r.dimension}: {r.segments_ranked} segments ranked | worst {r.worst} {r.worst_on_time_pct}% | "
          f"best {r.best} {r.best_on_time_pct}% | gap {r.gap_points} pts | below {TARGET_ON_TIME}% target: {r.below_target} | RED: {r.red}")
print()
print(f"Target used: {TARGET_ON_TIME}% (amber band {AMBER_BAND} pts), min orders per segment: {MIN_ORDERS}")
print("Avg profit per order, on time vs late:", late_vs_ontime.loc["on time", "avg_profit"], "vs", late_vs_ontime.loc["late", "avg_profit"])

## 8. Excel scorecard (with live formulas)
On-time % and the status colour are real Excel formulas that read the target from the **Settings** sheet. Change the target in Excel and every sheet updates.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.formatting.rule import CellIsRule

wb = Workbook()
ws = wb.active
ws.title = "Settings"
ws["A1"] = "On-time target (%)";                 ws["B1"] = TARGET_ON_TIME
ws["A2"] = "Amber band (points below target)";   ws["B2"] = AMBER_BAND
ws["A4"] = "Change B1 or B2 and the Status column on every sheet updates."
ws.column_dimensions["A"].width = 40
for c in ("A1", "A2"):
    ws[c].font = Font(bold=True)

HEAD_FILL = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
RAG_FILL = {"GREEN": "C6EFCE", "AMBER": "FFEB9C", "RED": "FFC7CE"}

for by, title in DIMS.items():
    sc = SC[by]
    w = wb.create_sheet(title)
    w.append([title, "Orders", "Late orders", "On-time %", "Avg ship days", "Avg scheduled days",
              "Avg delay when late (days)", "Sales", "Profit", "Status"])
    for i, r in enumerate(sc.to_dict("records"), start=2):
        delay = None if pd.isna(r["avg_delay_when_late"]) else r["avg_delay_when_late"]
        w.append([r[by], int(r["orders"]), int(r["late_orders"]),
                  f"=ROUND(100*(1-C{i}/B{i}),1)",
                  r["avg_ship_days"], r["avg_sched_days"], delay, r["sales"], r["profit"],
                  f'=IF(D{i}>=Settings!$B$1,"GREEN",IF(D{i}>=Settings!$B$1-Settings!$B$2,"AMBER","RED"))'])
    last = len(sc) + 1
    for name, colour in RAG_FILL.items():
        w.conditional_formatting.add(
            f"J2:J{last}",
            CellIsRule(operator="equal", formula=[f'"{name}"'],
                       fill=PatternFill(start_color=colour, end_color=colour, fill_type="solid")))
    for cell in w[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = HEAD_FILL
        cell.alignment = Alignment(horizontal="center", wrap_text=True)
    for col, width in zip("ABCDEFGHIJ", [26, 10, 12, 11, 13, 17, 20, 14, 14, 10]):
        w.column_dimensions[col].width = width
    for row in w.iter_rows(min_row=2, min_col=8, max_col=9):
        for cell in row:
            cell.number_format = "#,##0"
    w.freeze_panes = "A2"

wb.save("carrier_vendor_scorecard.xlsx")
print("saved carrier_vendor_scorecard.xlsx")

## 9. Exports for Tableau

In [ ]:
os.makedirs("tableau_exports", exist_ok=True)
orders.drop(columns=["order_date_raw"]).to_csv("tableau_exports/orders_clean.csv", index=False)
for by in DIMS:
    SC[by].to_csv(f"tableau_exports/scorecard_{by}.csv", index=False)
trend.to_csv("tableau_exports/monthly_on_time.csv", index=False)

import shutil
shutil.make_archive("tableau_exports", "zip", "tableau_exports")
try:
    from google.colab import files
    files.download("carrier_vendor_scorecard.xlsx")
    files.download("tableau_exports.zip")
except ImportError:
    print("Saved locally: carrier_vendor_scorecard.xlsx and tableau_exports.zip")

## 10. Tableau dashboard
Unzip `tableau_exports.zip`, then connect Tableau to `orders_clean.csv` (create an **extract**, which is needed to publish to Tableau Public). Build:
1. **KPI tiles:** on-time % (average of `is_late`, then 1 minus it as a calculated field `1 - AVG([Is Late])`), avg delay days when late.
2. **Bar by shipping mode:** on-time % with a reference line at your target.
3. **Heat map:** `order_region` (rows) by `shipping_mode` (columns), coloured by on-time %.
4. **Scorecard table** from `scorecard_order_region.csv`, coloured by `rag`.
5. **Monthly trend** of on-time %.
6. Filters: market, department, month.
Publish to Tableau Public and add the link to your README and resume.

## Optional: Excel pivot table
Open `orders_clean.csv` in Excel, then Insert, PivotTable. Put `shipping_mode` in Rows, `is_late` (Average) in Values, and `market` as a Filter. This shows Excel skills on top of the formula scorecard.